In [1]:
%pip install streamlit langchain langchain-community openai faiss-cpu unstructured requests beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


In [3]:
import streamlit as st
import pickle
import time
import langchain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import UnstructuredURLLoader
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader


In [8]:
loader=WebBaseLoader([
    "https://www.cnbc.com/quotes/TSLA","https://robinhood.com/us/en/stocks/TSLA/"
])
data=loader.load()

In [10]:
print(len(data))
print("\n")
print(data[0].page_content)
print("\n")
print(data[1].page_content)

2


TSLA: Tesla Inc - Stock Price, Quote and News - CNBCSkip NavigationMarketsPre-MarketsU.S. MarketsEurope MarketsChina MarketsAsia MarketsWorld MarketsCurrenciesPrediction MarketsCryptocurrencyFutures & CommoditiesBondsFunds & ETFsBusinessEconomyFinanceHealth & ScienceMediaReal EstateEnergyClimateTransportationInvestigationsIndustrialsRetailWealthSportsLifeSmall BusinessInvestingPersonal FinanceFintechFinancial AdvisorsOptions ActionETF StreetBuffett ArchiveEarningsTrader TalkTechCybersecurityAIEnterpriseInternetMediaMobileSocial MediaCNBC Disruptor 50Tech GuidePoliticsWhite HousePolicyDefenseCongressExpanding OpportunityEurope PoliticsChina PoliticsAsia PoliticsWorld PoliticsVideoLatest VideoFull EpisodesLivestreamTop VideoLive AudioEurope TVAsia TVCNBC PodcastsCEO InterviewsDigital OriginalsWatchlistInvesting ClubTrust PortfolioAnalysisTrade AlertsMeeting VideosHomestretchJim's ColumnsEducationSubscribePROPro NewsJosh BrownMike SantoliCalls of the DayMy PortfolioLivestreamFull Epis

Text Splitter


In [21]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=10
)
docs = splitter.split_documents(data)

In [22]:
len(docs)

69

In [23]:
docs[0]

Document(metadata={'source': 'https://www.cnbc.com/quotes/TSLA', 'title': 'TSLA: Tesla Inc - Stock Price, Quote and News - CNBC', 'description': 'Get Tesla Inc (TSLA:NASDAQ) real-time stock quotes, news, price and financial information from CNBC.', 'language': 'en'}, page_content='TSLA: Tesla Inc - Stock Price, Quote and News - CNBCSkip NavigationMarketsPre-MarketsU.S. MarketsEurope MarketsChina MarketsAsia MarketsWorld MarketsCurrenciesPrediction MarketsCryptocurrencyFutures &')

Embedding


In [16]:
pip install sentence-transformers langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [19]:
from dotenv import load_dotenv

load_dotenv()

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9237.33it/s]


FAISS DB


In [27]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9942.07it/s]


Create Retriever


In [28]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

LLM


In [ ]:
import os

os.environ['GROQ_API_KEY']="gsk_"

print(os.getenv("
                _API_KEY"))

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


In [2]:
from dotenv import load_dotenv
load_dotenv()

# os.environ['GROQ_API_KEY']=""

# print(os.getenv("GROQ_API_KEY"))

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)
print("Created LLM")

Created LLM


Prompt


In [ ]:
!pip install langchain.prom

In [48]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
    template="""
You are a financial news analyst.

Answer the question only using the provided context.

If the answer is not found in the context, say:
"I couldn't find that information in the provided articles."

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

RAG Pipeline


In [63]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain=(
    {'context':retriever|format_docs,
     'question':RunnablePassthrough()} | prompt | llm | StrOutputParser()
)

Ask


In [64]:
query = "What are analysts saying about Tesla stock?"

response = rag_chain.invoke(query)

print(response)

Analysts have given the following ratings for Tesla stock: 43.4% Buy, 43.4% Hold, and 13.2% Sell, based on 53 ratings.


With Sources


In [65]:
query = "What are analysts saying about Tesla stock?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\nSource {i+1}")
    print(doc.metadata)


Source 1
{'source': 'https://robinhood.com/us/en/stocks/TSLA/', 'title': 'Tesla: TSLA Stock Price Quote & News | Robinhood', 'description': 'View the real-time TSLA price chart on Robinhood and decide if you want to buy or sell commission-free. Other fees such as trading (non-commission) fees, Gold subscription fees, wire transfer fees, and paper statement fees may apply. See Robinhood Financial’s fee schedule at rbnhd.co/fees to learn more.', 'language': 'en'}

Source 2
{'source': 'https://www.cnbc.com/quotes/TSLA', 'title': 'TSLA: Tesla Inc - Stock Price, Quote and News - CNBC', 'description': 'Get Tesla Inc (TSLA:NASDAQ) real-time stock quotes, news, price and financial information from CNBC.', 'language': 'en'}

Source 3
{'source': 'https://robinhood.com/us/en/stocks/TSLA/', 'title': 'Tesla: TSLA Stock Price Quote & News | Robinhood', 'description': 'View the real-time TSLA price chart on Robinhood and decide if you want to buy or sell commission-free. Other fees such as trading (

Loop


In [ ]:
while True:
    question = input("\nAsk a question: ")

    if question.lower() == "exit":
        break

    answer = rag_chain.invoke(question)

    print("\nAnswer:")
    print(answer)